# Day 19 · A vector database vs. your NumPy: what does it add?

**First-principles question:** What problem does a vector database solve, and which problem doesn't it solve?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import json
import time
import numpy as np
import chromadb
from lab.search import Index, article_chunks

chunks = article_chunks()
index = Index(chunks)                      # your NumPy search
queries = [json.loads(l) for l in (ROOT / "data" / "acme" / "queries.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
client = chromadb.EphemeralClient()        # in memory: nothing is written to disk

## 1 · The same chunks in Chroma
You pass your own embeddings, so both searches see exactly the same vectors.

In [ ]:
# TODO: create a collection named "acme-kb" with cosine distance and no embedding function:
#   collection = client.get_or_create_collection("acme-kb", configuration={"hnsw": {"space": "cosine"}},
#                                                embedding_function=None)
# then collection.add(ids=..., embeddings=index.M.tolist(), documents=..., metadatas=[{"article": ...}, ...])
collection = None
print(collection.count(), "chunks in Chroma")

## 2 · Do they agree?

In [ ]:
from lab.llm import embed

def chroma_top(query, k=5):
    q = np.array(embed([query])[0]); q = q / np.linalg.norm(q)
    # TODO: collection.query(query_embeddings=[q.tolist()], n_results=k) and return the list of ids
    ...

def numpy_top(query, k=5):
    return [h["id"] for h in index.search(query, k=k)]

overlaps = []
for q in queries:
    a, b = set(chroma_top(q["query"])), set(numpy_top(q["query"]))
    overlaps.append(len(a & b) / 5)
print(f"average top-5 overlap: {np.mean(overlaps):.0%}")

## 3 · Speed at scale
20,000 random vectors: exact NumPy search vs. Chroma's approximate index.

In [ ]:
rng = np.random.default_rng(0)
big = rng.normal(size=(20_000, index.M.shape[1])).astype(np.float32)
big /= np.linalg.norm(big, axis=1, keepdims=True)

t0 = time.perf_counter()
big_col = client.get_or_create_collection("random-20k", configuration={"hnsw": {"space": "cosine"}},
                                          embedding_function=None)
for i in range(0, len(big), 5000):           # Chroma adds in batches
    big_col.add(ids=[f"r{j}" for j in range(i, i + 5000)], embeddings=big[i:i + 5000].tolist())
t_build = time.perf_counter() - t0

q = big[123]
t0 = time.perf_counter(); np.argsort(-(big @ q))[:5]; t_numpy = time.perf_counter() - t0
t0 = time.perf_counter(); big_col.query(query_embeddings=[q.tolist()], n_results=5); t_chroma = time.perf_counter() - t0
print(f"build index: {t_build:.1f} s · one query: numpy {t_numpy * 1000:.1f} ms, chroma {t_chroma * 1000:.1f} ms")

## 4 · What a vector database adds (and what it doesn't)
- adds: …
- adds: …
- adds: …
- does **not** fix: …

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
assert collection is not None and collection.count() == len(chunks), "Step 1: every chunk in Chroma"
assert np.mean(overlaps) >= 0.9, "Step 2: top-5 overlap should be at least 90% (same vectors, same distance)"
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")